# Prac 05

In this homework you are going to implement the **Floyd-Steinberg dithering** algorithm. Dithering, in general, means that we are adding noise to the signal (in our case digital image) in order to perceive it better. In other words, by adding the noise the objective quality will be worse but the subjective quality will be better (i.e. the image will "look" better).

The details of FS dithering can be found in this [wiki](https://en.wikipedia.org/wiki/Floyd%E2%80%93Steinberg_dithering) page. In order to implement the dithering, we will implement the following steps:
* Define colour pallette
* Quantize the image to obtain the baseline and compute the average quantization error
* Implement FS dithering and compute the average quantization error

You will also have to answer the question at the end of this notebook.

Note: In this homework, you will have the chance to earn some extra points. See the "Bonus" section at the end of the notebook. Good luck!

As always, you are encouraged to use your own images :-)

In [ ]:
import cv2
import math
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

In [ ]:
import urllib.request

url = 'https://res.cloudinary.com/jerrick/image/upload/v1719773902/6681aacefb8345001efd6e88.jpg'
urllib.request.urlretrieve(url, "queen.jpg")

Let's load the image.

In [ ]:
# Load image
img = cv2.imread('queen.jpg')
# Convert it to RGB
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# Plot it
plt.imshow(img)

Let's start with gray tones first.

In [ ]:
# Black, dark gray, light gray, white
colors = np.array([[0, 0, 0],
                   [64, 64, 64],
                   [192, 192, 192],
                   [255, 255, 255]])

Using the colour pallette, let's quantize the original image.

In [ ]:
# Cast the image to float
img = img.astype(float)

# Prepare for quantization
rows, cols, channels = img.shape
quantized = np.zeros_like(img)

# Apply quantization
for r in range(rows):
    for c in range(cols):
        # Extract the original pixel value
        pixel = img[r, c]

        # Find the closest colour from the pallette (using absolute value/Euclidean distance)
        # Note: You may need more than one line of code here
        new_pixel = min(colors, key=lambda x: np.linalg.norm(x - pixel))

        # Apply quantization
        quantized[r, c, :] = new_pixel

In [ ]:
# Show quantized image (don't forget to cast back to uint8)
plt.imshow(quantized.astype(np.uint8))

In [ ]:
# Compute average quantization error
avg_quant_error = np.abs(img - quantized).mean()

avg_quant_error

#### Floyd-Steinberg Dithering
We are now going to implement the FS dithering and compare it to the optimally quantized image we have calculated above.

In [ ]:
# Make a temporal copy of the original image, we will need it for error diffusion
img_tmp = np.copy(img)
dithering = np.zeros_like(img)

for r in range(1, rows-1):
    for c in range(1, cols-1):
        # Extract the original pixel value
        pixel = img_tmp[r, c]
        # Find the closest colour from the pallette (using absolute value/Euclidean distance)
        # Note: You may need more than one line of code here
        new_pixel = min(colors, key=lambda x: np.linalg.norm(x - pixel))

        # Compute quantization error
        quant_error = pixel - new_pixel

        # Diffuse the quantization error accroding to the FS diffusion matrix
        # Note: You may need more than one line of code here
        img_tmp[r, c + 1]     += quant_error * 7 / 16
        img_tmp[r + 1, c - 1] += quant_error * 3 / 16
        img_tmp[r + 1, c]     += quant_error * 5 / 16
        img_tmp[r + 1, c + 1] += quant_error * 1 / 16

        # Apply dithering
        dithering[r, c, :] = new_pixel

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))

plt.subplot(2, 1, 1)
plt.imshow(quantized.astype(np.uint8))
plt.title('Optimally Quantized Image', fontsize=14, pad=10)
plt.axis('off')


plt.subplot(2, 1, 2)
plt.imshow(dithering.astype(np.uint8))
plt.title('Image with Dithering Applied', fontsize=14, pad=10)
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Compute average quantization error for dithered image
avg_dith_error = np.abs(img - dithering).mean()
avg_dith_error

A loop usage will be very slow, so rewrite the code to take advantage of vectors procesing for the next interactions

In [ ]:
def get_opt_quant_img(img, colors):
    img_f = img.astype(float)
    colors_arr = np.array(colors)
    diff = img_f[:, :, np.newaxis, :] - colors_arr
    indices = np.argmin(np.sum(diff**2, axis=-1), axis=-1)
    quantized = colors_arr[indices]

    return quantized.astype(np.uint8)

def get_fs_dithered_img(img, colors):
    img_tmp = img.astype(float)
    colors_arr = np.array(colors)

    dithering = np.zeros_like(img_tmp)


    for r in range(1, rows - 1):
        for c in range(1, cols - 1):
            pixel = img_tmp[r, c]

            dists = np.sum((colors_arr - pixel) ** 2, axis=1)
            new_pixel = colors_arr[np.argmin(dists)]

            quant_error = pixel - new_pixel

            img_tmp[r,     c + 1] += quant_error * 7 / 16
            img_tmp[r + 1, c - 1] += quant_error * 3 / 16
            img_tmp[r + 1, c    ] += quant_error * 5 / 16
            img_tmp[r + 1, c + 1] += quant_error * 1 / 16

            dithering[r, c] = new_pixel

    return dithering.astype(np.uint8)

### Questions
* Which image has higher quantization error? Optimally quantized or dithered?
> Dithered
* Which image looks better to you?
> The FS dithered one
* Can you repeat the same process using only two colours: black and white? Show me :-)
> Just see below :-)

In [ ]:
colors = np.array([[0, 0, 0],
                   [255, 255, 255]])

opt_quant_img = get_opt_quant_img(img, colors)
fs_dithered_img = get_fs_dithered_img(img, colors)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))

plt.subplot(2, 1, 1)
plt.imshow(opt_quant_img.astype(np.uint8))
plt.title('Optimally Quantized', fontsize=14, pad=10)
plt.axis('off')


plt.subplot(2, 1, 2)
plt.imshow(fs_dithered_img.astype(np.uint8))
plt.title('Floyd-Steinberg Dithered', fontsize=14, pad=10)
plt.axis('off')

plt.tight_layout()
plt.show()

### Bonus Points

Repeat the homework using a diffrerent image pallette. For instance, you can use an optimal colour
pallette that we can calculate via k-means algorithm. The following snippet of code will give you the 16
optimal colours for your original image.

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=32).fit(np.reshape(img, (-1, 3)))
colors = kmeans.cluster_centers_

In [ ]:
opt_quant_img = get_opt_quant_img(img, colors)
fs_dithered_img = get_fs_dithered_img(img, colors)

In [ ]:
avg_quant_error = np.abs(img - opt_quant_img).mean()

avg_quant_error

In [ ]:
avg_dith_error = np.abs(img - fs_dithered_img).mean()
avg_dith_error

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 15))

plt.subplot(3, 1, 1)
plt.imshow(img.astype(np.uint8))
plt.title('Original Image', fontsize=14, fontweight='bold')
plt.axis('off')

plt.subplot(3, 1, 2)
plt.imshow(opt_quant_img.astype(np.uint8))
plt.title('Optimally Quantized', fontsize=14, fontweight='bold')
plt.axis('off')

plt.subplot(3, 1, 3)
plt.imshow(fs_dithered_img.astype(np.uint8))
plt.title('Floyd-Steinberg Dithered', fontsize=14, fontweight='bold')
plt.axis('off')

plt.tight_layout()
plt.show()

Apply FS dithering the same way you did before.
* How does the result look like to you?
> At a quick glance, it looks really good (in comparison to the orig) and very similar. The FS dithering algorithm is more enjoyable to the eye, because faces look brighter compared to the opt quant image (which had a grayish faces)
* What happens if we use 32 colours?
 > Obviously, the error decreases, and the difference between the orig and quantized image is already much harder to notice than before
* And what happens if we use 256 colours?
> Images will look very similar to the orig. If it was already difficult to differ at 32 colors, with 256 it becomes nearly impossible, and FS dithering becomes unnecessary at that point